# DEAP Subject-Dependent DTC Ablation

This notebook is self-contained and reuses the DEAP subject-dependent split setup from the MFMC fusion MLP reference experiment, while comparing three objective modes:

- `mfmc_dtc`: original cyclic pair-to-third DTC-grounded objective
- `pairwise_fmca`: tri-modal pairwise FMCA sum
- `shuffled_dtc`: cyclic pair-to-third objective with batch-wise rolled target modality

Outputs are saved under:

`/home/zhengdeyang/TAFFC_MFMC/MFMC/Supplement/DTC_ablation/results/`

Running the notebook top-to-bottom inside the `MFMC` conda environment will:

1. Load DEAP preprocessed data from the same path as the reference notebook.
2. Reuse the same backbone, classifier, batch size, optimizer defaults, iteration budget, and subject-dependent `StratifiedKFold` split logic.
   By default it only runs a single selected fold to save time.
3. Launch one full mode-per-GPU experiment when multiple GPUs are available.
4. Save per-fold curves/metrics, per-mode summaries, config JSON, aggregate CSV/PKL, and comparison plots for easy re-plotting later.


In [ ]:
import gc
import json
import math
import os
import pickle
import subprocess
import random
import threading
import time
import warnings
from dataclasses import dataclass
from datetime import timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Markdown, display
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")


In [ ]:
# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================

REPO_ROOT = Path(os.environ.get("TAFFC_MFMC_ROOT", "/home/zhengdeyang/TAFFC_MFMC"))
MFMC_ROOT = REPO_ROOT / "MFMC"
DATA_DIR = MFMC_ROOT / "DEAP" / "Data_processed"
RESULTS_ROOT = MFMC_ROOT / "Supplement" / "DTC_ablation" / "results"

NOTEBOOK_NAME = "DEAP_DTC_ablation_subject_dep"
MODES_TO_RUN = ["mfmc_dtc", "pairwise_fmca", "shuffled_dtc"]
USE_MULTIGPU = False
AUTO_SELECT_LARGEST_FREE_GPU = True
MANUAL_GPU_ID = None

BATCH_SIZE = 256
TOTAL_ITERATIONS = 10001
TRAINING_STEPS_PER_FOLD = TOTAL_ITERATIONS - 1
EVAL_INTERVAL = 500

REFERENCE_N_FOLDS = 5
ACTIVE_FOLDS = [1]
RANDOM_SEED = 42

LEARNING_RATE_ENCODER = 0.0003
LEARNING_RATE_CLASSIFIER = 0.0003
BETA1 = 0.9
BETA2 = 0.999
COV_BETA = 0.5
USE_CLASS_BALANCING = False
CLASSIFIER_EVAL_BATCH_SIZE = 100

FEATURE_DIM = 128
LOSS_EPS = 1e-6
MODE_COLORS = {
    "mfmc_dtc": "#1b9e77",
    "pairwise_fmca": "#d95f02",
    "shuffled_dtc": "#7570b3",
}

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def select_device(preferred_gpu_id=None) -> torch.device:
    if torch.cuda.is_available():
        if preferred_gpu_id is not None and 0 <= preferred_gpu_id < torch.cuda.device_count():
            return torch.device(f"cuda:{preferred_gpu_id}")
        return torch.device("cuda")
    return torch.device("cpu")

def detect_best_gpu_id():
    if not torch.cuda.is_available():
        return None, []

    try:
        result = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=index,memory.free,memory.total",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        )
        gpu_stats = []
        for raw_line in result.strip().splitlines():
            gpu_id_str, free_mem_str, total_mem_str = [item.strip() for item in raw_line.split(",")]
            gpu_stats.append({
                "gpu_id": int(gpu_id_str),
                "free_memory_mb": int(free_mem_str),
                "total_memory_mb": int(total_mem_str),
            })
        gpu_stats.sort(key=lambda row: (row["free_memory_mb"], row["total_memory_mb"], -row["gpu_id"]), reverse=True)
        return gpu_stats[0]["gpu_id"], gpu_stats
    except Exception:
        gpu_stats = []
        for gpu_id in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(gpu_id)
            gpu_stats.append({
                "gpu_id": gpu_id,
                "free_memory_mb": int(props.total_memory // (1024 ** 2)),
                "total_memory_mb": int(props.total_memory // (1024 ** 2)),
            })
        gpu_stats.sort(key=lambda row: (row["free_memory_mb"], row["total_memory_mb"], -row["gpu_id"]), reverse=True)
        return gpu_stats[0]["gpu_id"] if gpu_stats else None, gpu_stats


def resolve_gpu_id():
    if not torch.cuda.is_available():
        return None, []
    if AUTO_SELECT_LARGEST_FREE_GPU:
        return detect_best_gpu_id()
    return MANUAL_GPU_ID, []


def ensure_jsonable(value):
    if isinstance(value, dict):
        return {str(k): ensure_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [ensure_jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, torch.device):
        return str(value)
    return value

RUN_CONFIG = {
    "notebook": NOTEBOOK_NAME,
    "data_dir": DATA_DIR,
    "results_root": RESULTS_ROOT,
    "modes_to_run": MODES_TO_RUN,
    "use_multigpu": USE_MULTIGPU,
    "auto_select_largest_free_gpu": AUTO_SELECT_LARGEST_FREE_GPU,
    "manual_gpu_id": MANUAL_GPU_ID,
    "batch_size": BATCH_SIZE,
    "total_iterations": TOTAL_ITERATIONS,
    "training_steps_per_fold": TRAINING_STEPS_PER_FOLD,
    "eval_interval": EVAL_INTERVAL,
    "reference_n_folds": REFERENCE_N_FOLDS,
    "active_folds": ACTIVE_FOLDS,
    "active_fold_count": len(ACTIVE_FOLDS),
    "random_seed": RANDOM_SEED,
    "learning_rate_encoder": LEARNING_RATE_ENCODER,
    "learning_rate_classifier": LEARNING_RATE_CLASSIFIER,
    "beta1": BETA1,
    "beta2": BETA2,
    "cov_beta": COV_BETA,
    "use_class_balancing": USE_CLASS_BALANCING,
    "classifier_eval_batch_size": CLASSIFIER_EVAL_BATCH_SIZE,
    "feature_dim": FEATURE_DIM,
    "loss_eps": LOSS_EPS,
}

set_seed(RANDOM_SEED)
RESOLVED_GPU_ID, GPU_DETECTION_INFO = resolve_gpu_id()

RUN_CONFIG["resolved_gpu_id"] = RESOLVED_GPU_ID
RUN_CONFIG["gpu_detection_info"] = GPU_DETECTION_INFO

print("Configuration loaded successfully.")
print(f"Data directory: {DATA_DIR}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Modes: {MODES_TO_RUN}")
print(f"Auto select largest-free GPU: {AUTO_SELECT_LARGEST_FREE_GPU}")
print(f"Manual GPU id override: {MANUAL_GPU_ID}")
print(f"Resolved GPU id: {RESOLVED_GPU_ID}")
print(f"Available CUDA devices: {torch.cuda.device_count()}")
if GPU_DETECTION_INFO:
    print("GPU detection info (sorted by chosen priority):")
    display(pd.DataFrame(GPU_DETECTION_INFO))


In [ ]:
# =============================================================================
# MODEL DEFINITIONS
# =============================================================================

class NETWORK_F_MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super().__init__()
        self.dim = out_dim
        self.num_layers = num_layers
        self.fc_list = []
        self.bn_list = []

        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        for _ in range(self.num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        self.fc_list = nn.ModuleList(self.fc_list)
        self.bn_list = nn.ModuleList(self.bn_list)
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        for i in range(self.num_layers):
            x = self.fc_list[i](x)
            x = torch.relu(x)
            x = self.bn_list[i](x)
        x = self.fc_final(x)
        x = torch.sigmoid(x)
        return x


class Advanced1DCNN_channel(nn.Module):
    def __init__(self, input_channels=1, num_classes=128, input_size=1280):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )

        feat_size = input_size // (4 * 4 * 4 * 4)
        self.fc1 = nn.Sequential(
            nn.Linear(256 * feat_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
        )
        self.fc3 = nn.Linear(512, num_classes)
        self.MLP = NETWORK_F_MLP(
            input_dim=128 * input_channels,
            hidden_dim=4000,
            out_dim=num_classes,
            num_layers=1,
        )

    def forward(self, x):
        batch_size, channels = x.shape[0], x.shape[1]
        x = x.unsqueeze(2)
        x = x.flatten(0, 1)

        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)

        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)

        out = out.reshape(batch_size, channels, -1)
        out = out.flatten(-2, -1)
        out = self.MLP(out)
        return out


class ComplexClassifier(nn.Module):
    def __init__(self, dim_features=128, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(dim_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x


class ProjectionHead(nn.Module):
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return x


def build_models(device):
    models = {
        "eeg_encoder": Advanced1DCNN_channel(
            input_channels=eeg_data.shape[1], num_classes=FEATURE_DIM, input_size=eeg_data.shape[2]
        ).to(device),
        "eog_encoder": Advanced1DCNN_channel(
            input_channels=eog_data.shape[1], num_classes=FEATURE_DIM, input_size=eog_data.shape[2]
        ).to(device),
        "temp_encoder": Advanced1DCNN_channel(
            input_channels=temp_data.shape[1], num_classes=FEATURE_DIM, input_size=temp_data.shape[2]
        ).to(device),
        "proj_12": ProjectionHead(input_dim=256, hidden_dim=512, output_dim=FEATURE_DIM).to(device),
        "proj_13": ProjectionHead(input_dim=256, hidden_dim=512, output_dim=FEATURE_DIM).to(device),
        "proj_23": ProjectionHead(input_dim=256, hidden_dim=512, output_dim=FEATURE_DIM).to(device),
        "classifier": ComplexClassifier(
            dim_features=FEATURE_DIM,
            num_classes=len(torch.unique(emotion_labels)),
        ).to(device),
    }
    return models


In [ ]:
# =============================================================================
# MFMC / DTC OBJECTIVES
# =============================================================================

@dataclass
class ObjectiveComputation:
    loss: torch.Tensor
    terms: dict


def adaptive_estimation(v_t, beta, square_term, i):
    v_t = beta * v_t + (1 - beta) * square_term.detach()
    return v_t, (v_t / (1 - beta ** i))


def MFMC_t_trace(x, y, track_cov, i, cov_beta=0.95):
    Rx = (x.T @ x) / x.shape[0]
    Ry = (y.T @ y) / y.shape[0]
    Pxy = (x.T @ y) / x.shape[0]

    Rx = Rx + torch.eye(Rx.shape[0], device=Rx.device, dtype=Rx.dtype) * LOSS_EPS
    Ry = Ry + torch.eye(Ry.shape[0], device=Ry.device, dtype=Ry.dtype) * LOSS_EPS

    track_cov["Rx"], Rx_est = adaptive_estimation(track_cov["Rx"], cov_beta, Rx, i)
    track_cov["Ry"], Ry_est = adaptive_estimation(track_cov["Ry"], cov_beta, Ry, i)
    track_cov["Pxy"], Pxy_est = adaptive_estimation(track_cov["Pxy"], cov_beta, Pxy, i)

    Rx_est_inv = torch.inverse(Rx_est)
    Ry_est_inv = torch.inverse(Ry_est)

    cost = (
        -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T
        + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T
        - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T
        + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T
    )

    loss = -torch.trace(cost)
    return track_cov, loss


def init_trackers(device, mode, feature_dim=FEATURE_DIM):
    if mode in {"mfmc_dtc", "shuffled_dtc"}:
        keys = ["track_1_23", "track_2_13", "track_3_12"]
    elif mode == "pairwise_fmca":
        keys = ["track_12", "track_13", "track_23"]
    else:
        raise ValueError(f"Unsupported mode: {mode}")

    return {
        key: {
            "Rx": torch.zeros(feature_dim, feature_dim, device=device),
            "Ry": torch.zeros(feature_dim, feature_dim, device=device),
            "Pxy": torch.zeros(feature_dim, feature_dim, device=device),
        }
        for key in keys
    }


def mfmc_dtc_loss(fe1, fe2, fe3, proj12_head, proj13_head, proj23_head, trackers, step, cov_beta):
    proj_12 = proj12_head(torch.cat([fe1, fe2], dim=1))
    proj_13 = proj13_head(torch.cat([fe1, fe3], dim=1))
    proj_23 = proj23_head(torch.cat([fe2, fe3], dim=1))

    trackers["track_1_23"], loss1 = MFMC_t_trace(fe1, proj_23, trackers["track_1_23"], step, cov_beta)
    trackers["track_2_13"], loss2 = MFMC_t_trace(fe2, proj_13, trackers["track_2_13"], step, cov_beta)
    trackers["track_3_12"], loss3 = MFMC_t_trace(fe3, proj_12, trackers["track_3_12"], step, cov_beta)

    return trackers, ObjectiveComputation(
        loss=loss1 + loss2 + loss3,
        terms={
            "loss_1_23": float(loss1.detach().cpu()),
            "loss_2_13": float(loss2.detach().cpu()),
            "loss_3_12": float(loss3.detach().cpu()),
        },
    )


def pairwise_fmca_loss(fe1, fe2, fe3, trackers, step, cov_beta):
    trackers["track_12"], loss12 = MFMC_t_trace(fe1, fe2, trackers["track_12"], step, cov_beta)
    trackers["track_13"], loss13 = MFMC_t_trace(fe1, fe3, trackers["track_13"], step, cov_beta)
    trackers["track_23"], loss23 = MFMC_t_trace(fe2, fe3, trackers["track_23"], step, cov_beta)

    return trackers, ObjectiveComputation(
        loss=loss12 + loss13 + loss23,
        terms={
            "loss_12": float(loss12.detach().cpu()),
            "loss_13": float(loss13.detach().cpu()),
            "loss_23": float(loss23.detach().cpu()),
        },
    )


def shuffled_dtc_loss(fe1, fe2, fe3, proj12_head, proj13_head, proj23_head, trackers, step, cov_beta):
    proj_12 = proj12_head(torch.cat([fe1, fe2], dim=1))
    proj_13 = proj13_head(torch.cat([fe1, fe3], dim=1))
    proj_23 = proj23_head(torch.cat([fe2, fe3], dim=1))

    batch_size = fe1.shape[0]
    perm = torch.roll(torch.arange(batch_size, device=fe1.device), shifts=1)

    trackers["track_1_23"], loss1 = MFMC_t_trace(fe1[perm], proj_23, trackers["track_1_23"], step, cov_beta)
    trackers["track_2_13"], loss2 = MFMC_t_trace(fe2[perm], proj_13, trackers["track_2_13"], step, cov_beta)
    trackers["track_3_12"], loss3 = MFMC_t_trace(fe3[perm], proj_12, trackers["track_3_12"], step, cov_beta)

    return trackers, ObjectiveComputation(
        loss=loss1 + loss2 + loss3,
        terms={
            "loss_1_23": float(loss1.detach().cpu()),
            "loss_2_13": float(loss2.detach().cpu()),
            "loss_3_12": float(loss3.detach().cpu()),
        },
    )


def compute_objective(mode, fe1, fe2, fe3, models, trackers, step, cov_beta=COV_BETA):
    if mode == "mfmc_dtc":
        return mfmc_dtc_loss(
            fe1, fe2, fe3,
            models["proj_12"], models["proj_13"], models["proj_23"],
            trackers, step, cov_beta,
        )
    if mode == "pairwise_fmca":
        return pairwise_fmca_loss(fe1, fe2, fe3, trackers, step, cov_beta)
    if mode == "shuffled_dtc":
        return shuffled_dtc_loss(
            fe1, fe2, fe3,
            models["proj_12"], models["proj_13"], models["proj_23"],
            trackers, step, cov_beta,
        )
    raise ValueError(f"Unknown mode: {mode}")


In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

print("Loading DEAP dataset...")
print(f"Loading data from: {DATA_DIR}")

subject = torch.from_numpy(np.load(DATA_DIR / "subject.npy")).long()
emotion_labels = torch.from_numpy(np.load(DATA_DIR / "emotion_labels.npy")).long()
eeg_data = torch.from_numpy(np.load(DATA_DIR / "eeg_data.npy")).float()
eog_data = torch.from_numpy(np.load(DATA_DIR / "eog_data.npy")).float()
temp_data = torch.from_numpy(np.load(DATA_DIR / "temp_data.npy")).float()

print("Data loaded successfully.")
print(f"Total samples: {eeg_data.shape[0]}")
print(f"EEG shape: {tuple(eeg_data.shape)}")
print(f"EOG shape: {tuple(eog_data.shape)}")
print(f"TEMP shape: {tuple(temp_data.shape)}")
print(f"Labels shape: {tuple(emotion_labels.shape)}")
print(f"Unique subjects: {len(torch.unique(subject))}")
print(f"Emotion classes: {len(torch.unique(emotion_labels))}")

class_counts = torch.bincount(emotion_labels)
quadrant_names = [
    "Low V-Low A (Sad)",
    "Low V-High A (Angry)",
    "High V-Low A (Calm)",
    "High V-High A (Happy)",
]
for idx, (name, count) in enumerate(zip(quadrant_names, class_counts.tolist())):
    print(f"Class {idx} - {name}: {count} samples ({count / len(emotion_labels) * 100:.1f}%)")

indices = np.arange(eeg_data.shape[0])
skf = StratifiedKFold(n_splits=REFERENCE_N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
all_fold_splits = []
for fold_idx, (train_indices, test_indices) in enumerate(skf.split(indices, emotion_labels.numpy()), start=1):
    all_fold_splits.append(
        {
            "fold": fold_idx,
            "train_indices": train_indices,
            "test_indices": test_indices,
            "train_size": len(train_indices),
            "test_size": len(test_indices),
        }
    )
    print(f"Fold {fold_idx}: Train={len(train_indices)}, Test={len(test_indices)}")
fold_splits = [fold_info for fold_info in all_fold_splits if fold_info["fold"] in ACTIVE_FOLDS]
print(f"Selected folds to run: {ACTIVE_FOLDS}")
print(f"Actual fold count to run: {len(fold_splits)}")


In [ ]:
# =============================================================================
# TRAINING / EVALUATION / SAVING UTILITIES
# =============================================================================

MODEL_INIT_LOCK = threading.Lock()


def mode_dir(mode: str) -> Path:
    path = RESULTS_ROOT / mode
    path.mkdir(parents=True, exist_ok=True)
    return path


def build_fold_tensors(fold_info):
    train_idx = fold_info["train_indices"]
    test_idx = fold_info["test_indices"]
    return {
        "train_eeg": eeg_data[train_idx],
        "test_eeg": eeg_data[test_idx],
        "train_eog": eog_data[train_idx],
        "test_eog": eog_data[test_idx],
        "train_temp": temp_data[train_idx],
        "test_temp": temp_data[test_idx],
        "train_labels": emotion_labels[train_idx],
        "test_labels": emotion_labels[test_idx],
    }


def build_classifier_criterion(train_labels, device):
    if USE_CLASS_BALANCING:
        class_counts = torch.bincount(train_labels)
        class_weights = 1.0 / class_counts.float()
        class_weights = class_weights / class_weights.sum() * len(class_weights)
        return nn.CrossEntropyLoss(weight=class_weights.to(device))
    return nn.CrossEntropyLoss()


def evaluate_eeg_classifier(models, test_eeg, test_labels, device):
    models["eeg_encoder"].eval()
    models["classifier"].eval()
    all_true = []
    all_pred = []

    with torch.no_grad():
        for start in range(0, len(test_eeg), CLASSIFIER_EVAL_BATCH_SIZE):
            end = min(start + CLASSIFIER_EVAL_BATCH_SIZE, len(test_eeg))
            eeg_batch = test_eeg[start:end].to(device)
            label_batch = test_labels[start:end].to(device)
            logits = models["classifier"](models["eeg_encoder"](eeg_batch))
            pred = torch.argmax(logits, dim=1)
            all_true.extend(label_batch.cpu().numpy().tolist())
            all_pred.extend(pred.cpu().numpy().tolist())

    acc = accuracy_score(all_true, all_pred)
    macro_f1 = f1_score(all_true, all_pred, average="macro")
    models["eeg_encoder"].train()
    models["classifier"].train()
    return acc, macro_f1, all_true, all_pred


def feature_optimizer_params(models):
    return (
        list(models["eeg_encoder"].parameters())
        + list(models["eog_encoder"].parameters())
        + list(models["temp_encoder"].parameters())
        + list(models["proj_12"].parameters())
        + list(models["proj_13"].parameters())
        + list(models["proj_23"].parameters())
    )


def pad_last(values, target_len):
    if not values:
        return [np.nan] * target_len
    if len(values) >= target_len:
        return values[:target_len]
    return values + [values[-1]] * (target_len - len(values))


def save_json(path: Path, payload):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(ensure_jsonable(payload), f, indent=2)


def save_pickle(path: Path, payload):
    with open(path, "wb") as f:
        pickle.dump(payload, f)


def plot_fold_learning_curves(curve_df, save_path, mode, fold):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    color = MODE_COLORS[mode]

    axes[0, 0].plot(curve_df["iteration"], curve_df["mfmc_loss"], color=color)
    axes[0, 0].set_title(f"{mode} - Fold {fold} objective loss")
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("Loss")

    axes[0, 1].plot(curve_df["iteration"], curve_df["classifier_loss"], color="#444444")
    axes[0, 1].set_title(f"{mode} - Fold {fold} classifier loss")
    axes[0, 1].set_xlabel("Iteration")
    axes[0, 1].set_ylabel("CrossEntropy")

    eval_df = curve_df.dropna(subset=["eval_accuracy"]).copy()
    axes[1, 0].plot(eval_df["iteration"], eval_df["eval_accuracy"], marker="o", color=color)
    axes[1, 0].set_title(f"{mode} - Fold {fold} accuracy")
    axes[1, 0].set_xlabel("Iteration")
    axes[1, 0].set_ylabel("Accuracy")

    axes[1, 1].plot(eval_df["iteration"], eval_df["eval_macro_f1"], marker="o", color="#c44e52")
    axes[1, 1].set_title(f"{mode} - Fold {fold} macro-F1")
    axes[1, 1].set_xlabel("Iteration")
    axes[1, 1].set_ylabel("Macro-F1")

    for ax in axes.ravel():
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def aggregate_curve_metrics(fold_curve_dfs, metric_names):
    max_len = max(len(df) for df in fold_curve_dfs)
    agg = {}
    for metric in metric_names:
        series_list = []
        for df in fold_curve_dfs:
            values = df[metric].tolist()
            series_list.append(pad_last(values, max_len))
        metric_array = np.asarray(series_list, dtype=float)
        agg[metric] = {
            "mean": np.nanmean(metric_array, axis=0),
            "std": np.nanstd(metric_array, axis=0),
        }
    return np.arange(1, max_len + 1), agg


def plot_mode_learning_curves(mode, fold_curve_dfs, save_path):
    metric_names = ["mfmc_loss", "classifier_loss", "eval_accuracy", "eval_macro_f1"]
    x, agg = aggregate_curve_metrics(fold_curve_dfs, metric_names)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    layout = [
        ("mfmc_loss", "Objective loss", MODE_COLORS[mode]),
        ("classifier_loss", "Classifier loss", "#444444"),
        ("eval_accuracy", "Accuracy", MODE_COLORS[mode]),
        ("eval_macro_f1", "Macro-F1", "#c44e52"),
    ]

    for ax, (metric, title, color) in zip(axes.ravel(), layout):
        mean = agg[metric]["mean"]
        std = agg[metric]["std"]
        ax.plot(x, mean, color=color)
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.2)
        ax.set_title(f"{mode} - {len(fold_curve_dfs)}-fold mean ± std {title}")
        ax.set_xlabel("Recorded step")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_mode_triptych(mode, fold_curve_dfs, save_path):
    metric_names = ["mfmc_loss", "classifier_loss", "eval_accuracy"]
    x, agg = aggregate_curve_metrics(fold_curve_dfs, metric_names)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    layout = [
        ("mfmc_loss", "Cost", MODE_COLORS[mode]),
        ("classifier_loss", "Loss", "#444444"),
        ("eval_accuracy", "Test Acc", MODE_COLORS[mode]),
    ]

    for ax, (metric, title, color) in zip(axes, layout):
        mean = agg[metric]["mean"]
        std = agg[metric]["std"]
        ax.plot(x, mean, color=color, linewidth=2)
        ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.2)
        ax.set_title(f"{mode} - {title}")
        ax.set_xlabel("Recorded step")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_mode_comparison(summary_df, save_path):
    metrics = [
        ("best_accuracy_mean", "Best Accuracy"),
        ("best_macro_f1_mean", "Best Macro-F1"),
        ("final_accuracy_mean", "Final Accuracy"),
        ("final_macro_f1_mean", "Final Macro-F1"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    for ax, (metric, title) in zip(axes.ravel(), metrics):
        err_col = metric.replace("_mean", "_std")
        ax.bar(
            summary_df["mode"],
            summary_df[metric],
            yerr=summary_df[err_col],
            color=[MODE_COLORS[m] for m in summary_df["mode"]],
            alpha=0.85,
            capsize=5,
        )
        ax.set_title(title)
        ax.set_ylabel(title)
        ax.set_xlabel("Mode")
        ax.tick_params(axis="x", rotation=15)
        ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def save_mode_summary_artifacts(mode, mode_summary, fold_curve_dfs):
    mdir = mode_dir(mode)
    pd.DataFrame(mode_summary["fold_results"]).to_csv(mdir / "fold_metrics.csv", index=False)
    pd.DataFrame([mode_summary["summary"]]).to_csv(mdir / "summary.csv", index=False)
    save_json(mdir / "summary.json", mode_summary)
    save_pickle(mdir / "raw_results.pkl", mode_summary)
    plot_mode_learning_curves(mode, fold_curve_dfs, mdir / "learning_curves_mean_std.png")
    plot_mode_triptych(mode, fold_curve_dfs, mdir / "learning_curves_triptych.png")


def run_one_fold(mode, fold_info, run_device, mode_output_dir, show_progress=True):
    fold_num = fold_info["fold"]
    fold_tensors = build_fold_tensors(fold_info)
    criterion = build_classifier_criterion(fold_tensors["train_labels"], run_device)

    with MODEL_INIT_LOCK:
        set_seed(RANDOM_SEED + fold_num)
        models = build_models(run_device)

    optimizer_features = optim.Adam(
        feature_optimizer_params(models),
        lr=LEARNING_RATE_ENCODER,
        betas=(BETA1, BETA2),
        amsgrad=True,
    )
    optimizer_classifier = optim.Adam(
        models["classifier"].parameters(),
        lr=LEARNING_RATE_CLASSIFIER,
        betas=(BETA1, BETA2),
        amsgrad=True,
    )
    trackers = init_trackers(run_device, mode)

    curve_rows = []
    best_accuracy = -np.inf
    best_macro_f1 = -np.inf
    final_accuracy = np.nan
    final_macro_f1 = np.nan
    best_iteration = None
    last_y_true = []
    last_y_pred = []

    fold_start = time.time()

    epoch_bar = tqdm(
        total=TRAINING_STEPS_PER_FOLD,
        desc=f"{mode} epochs",
        leave=True,
        dynamic_ncols=True,
        disable=not show_progress,
    )

    for iteration in range(1, TOTAL_ITERATIONS):
        optimizer_features.zero_grad(set_to_none=True)

        batch_indices = torch.randint(0, len(fold_tensors["train_eeg"]), (BATCH_SIZE,))
        input_eeg = fold_tensors["train_eeg"][batch_indices].to(run_device)
        input_eog = fold_tensors["train_eog"][batch_indices].to(run_device)
        input_temp = fold_tensors["train_temp"][batch_indices].to(run_device)

        fe1 = models["eeg_encoder"](input_eeg)
        fe2 = models["eog_encoder"](input_eog)
        fe3 = models["temp_encoder"](input_temp)

        trackers, objective = compute_objective(mode, fe1, fe2, fe3, models, trackers, iteration, COV_BETA)
        objective.loss.backward()
        optimizer_features.step()

        optimizer_classifier.zero_grad(set_to_none=True)
        batch_indices = torch.randint(0, len(fold_tensors["train_eeg"]), (BATCH_SIZE,))
        input_eeg = fold_tensors["train_eeg"][batch_indices].to(run_device)
        labels_batch = fold_tensors["train_labels"][batch_indices].to(run_device)

        with torch.no_grad():
            feature_eeg = models["eeg_encoder"](input_eeg)
        logits = models["classifier"](feature_eeg.detach())
        classifier_loss = criterion(logits, labels_batch)
        classifier_loss.backward()
        optimizer_classifier.step()

        row = {
            "mode": mode,
            "fold": fold_num,
            "iteration": iteration,
            "mfmc_loss": float(objective.loss.detach().cpu()),
            "classifier_loss": float(classifier_loss.detach().cpu()),
            "eval_accuracy": np.nan,
            "eval_macro_f1": np.nan,
        }
        row.update(objective.terms)

        if iteration % EVAL_INTERVAL == 0:
            acc, macro_f1, y_true, y_pred = evaluate_eeg_classifier(
                models,
                fold_tensors["test_eeg"],
                fold_tensors["test_labels"],
                run_device,
            )
            row["eval_accuracy"] = float(acc)
            row["eval_macro_f1"] = float(macro_f1)
            final_accuracy = float(acc)
            final_macro_f1 = float(macro_f1)
            last_y_true = y_true
            last_y_pred = y_pred
            if acc > best_accuracy:
                best_accuracy = float(acc)
            if macro_f1 > best_macro_f1:
                best_macro_f1 = float(macro_f1)
            if best_iteration is None or acc >= best_accuracy:
                best_iteration = iteration

        curve_rows.append(row)

        epoch_bar.update(1)

        if iteration % EVAL_INTERVAL == 0:
            epoch_bar.set_postfix({
                "epoch": iteration,
                "obj": f"{row['mfmc_loss']:.4f}",
                "cls": f"{row['classifier_loss']:.4f}",
                "acc": f"{row['eval_accuracy']:.4f}",
                "f1": f"{row['eval_macro_f1']:.4f}",
            })

    epoch_bar.close()

    curve_df = pd.DataFrame(curve_rows)
    curve_path = mode_output_dir / f"fold_{fold_num}_training_curve.csv"
    curve_df.to_csv(curve_path, index=False)
    plot_fold_learning_curves(curve_df, mode_output_dir / f"fold_{fold_num}_learning_curves.png", mode, fold_num)

    predictions_df = pd.DataFrame({"y_true": last_y_true, "y_pred": last_y_pred})
    predictions_df.to_csv(mode_output_dir / f"fold_{fold_num}_predictions.csv", index=False)

    fold_time = time.time() - fold_start
    fold_result = {
        "mode": mode,
        "fold": fold_num,
        "train_size": int(fold_info["train_size"]),
        "test_size": int(fold_info["test_size"]),
        "best_accuracy": float(best_accuracy),
        "best_macro_f1": float(best_macro_f1),
        "final_accuracy": float(final_accuracy),
        "final_macro_f1": float(final_macro_f1),
        "best_iteration": int(best_iteration if best_iteration is not None else -1),
        "training_time_seconds": float(fold_time),
        "curve_csv": str(curve_path),
    }
    save_json(mode_output_dir / f"fold_{fold_num}_metrics.json", fold_result)

    del models, optimizer_features, optimizer_classifier, trackers
    gc.collect()
    if run_device.type == "cuda":
        torch.cuda.empty_cache()

    return fold_result, curve_df


In [ ]:
# =============================================================================
# MODE-LEVEL SEQUENTIAL EXECUTION HELPERS
# =============================================================================

GLOBAL_MODE_RESULTS = {}
RUN_DEVICE = None


def get_run_device(gpu_id=RESOLVED_GPU_ID):
    run_device = select_device(gpu_id)
    if run_device.type == "cuda":
        torch.cuda.set_device(run_device)
    return run_device


def run_single_mode(mode, gpu_id=RESOLVED_GPU_ID, show_progress=True):
    run_device = get_run_device(gpu_id)
    mdir = mode_dir(mode)
    fold_results = []
    fold_curve_dfs = []
    total_start = time.time()

    for fold_info in fold_splits:
        fold_result, curve_df = run_one_fold(
            mode,
            fold_info,
            run_device,
            mdir,
            show_progress=show_progress,
        )
        fold_results.append(fold_result)
        fold_curve_dfs.append(curve_df)

    summary = {
        "mode": mode,
        "device": str(run_device),
        "gpu_id": None if run_device.type != "cuda" else run_device.index,
        "best_accuracy_mean": float(np.mean([r["best_accuracy"] for r in fold_results])),
        "best_accuracy_std": float(np.std([r["best_accuracy"] for r in fold_results])),
        "best_macro_f1_mean": float(np.mean([r["best_macro_f1"] for r in fold_results])),
        "best_macro_f1_std": float(np.std([r["best_macro_f1"] for r in fold_results])),
        "final_accuracy_mean": float(np.mean([r["final_accuracy"] for r in fold_results])),
        "final_accuracy_std": float(np.std([r["final_accuracy"] for r in fold_results])),
        "final_macro_f1_mean": float(np.mean([r["final_macro_f1"] for r in fold_results])),
        "final_macro_f1_std": float(np.std([r["final_macro_f1"] for r in fold_results])),
        "total_training_time_seconds": float(time.time() - total_start),
    }

    mode_summary = {
        "mode": mode,
        "summary": summary,
        "fold_results": fold_results,
        "config": RUN_CONFIG,
    }
    save_mode_summary_artifacts(mode, mode_summary, fold_curve_dfs)

    GLOBAL_MODE_RESULTS[mode] = {
        "mode_summary": mode_summary,
        "fold_curve_dfs": fold_curve_dfs,
    }

    if run_device.type == "cuda":
        torch.cuda.empty_cache()

    return mode_summary, fold_curve_dfs


def summarize_completed_modes(modes=None):
    completed_modes = modes or [mode for mode in MODES_TO_RUN if mode in GLOBAL_MODE_RESULTS]
    if not completed_modes:
        print("No completed mode results found yet.")
        return None

    summary_rows = [GLOBAL_MODE_RESULTS[mode]["mode_summary"]["summary"] for mode in completed_modes]
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.set_index("mode").loc[completed_modes].reset_index()
    return summary_df


def save_aggregate_artifacts(modes=None):
    completed_modes = modes or [mode for mode in MODES_TO_RUN if mode in GLOBAL_MODE_RESULTS]
    if not completed_modes:
        print("No completed mode results to aggregate.")
        return None

    summary_df = summarize_completed_modes(completed_modes)
    all_mode_summaries = {
        mode: GLOBAL_MODE_RESULTS[mode]["mode_summary"]
        for mode in completed_modes
    }

    for mode, payload in all_mode_summaries.items():
        pd.DataFrame(payload["fold_results"]).to_csv(
            RESULTS_ROOT / f"{mode}_fold_metrics.csv",
            index=False,
        )

    summary_csv_path = RESULTS_ROOT / "mode_summary.csv"
    summary_df.to_csv(summary_csv_path, index=False)

    all_fold_rows = []
    for mode in completed_modes:
        all_fold_rows.extend(all_mode_summaries[mode]["fold_results"])
    all_fold_df = pd.DataFrame(all_fold_rows)
    all_fold_csv_path = RESULTS_ROOT / "all_fold_metrics.csv"
    all_fold_df.to_csv(all_fold_csv_path, index=False)

    comparison_plot_path = RESULTS_ROOT / "mode_comparison_barplot.png"
    plot_mode_comparison(summary_df, comparison_plot_path)

    aggregate_payload = {
        "config": RUN_CONFIG,
        "completed_modes": completed_modes,
        "summary_df": summary_df.to_dict(orient="records"),
        "all_mode_summaries": all_mode_summaries,
    }
    save_pickle(RESULTS_ROOT / "all_mode_results.pkl", aggregate_payload)
    save_json(RESULTS_ROOT / "all_mode_results.json", aggregate_payload)

    print("=" * 80)
    print("DEAP subject-dependent DTC ablation aggregation finished.")
    print(f"Completed modes: {completed_modes}")
    print(f"Summary CSV: {summary_csv_path}")
    print(f"All-fold CSV: {all_fold_csv_path}")
    print(f"Comparison plot: {comparison_plot_path}")
    print(f"Aggregate PKL: {RESULTS_ROOT / 'all_mode_results.pkl'}")
    print("=" * 80)
    return summary_df


In [ ]:
# =============================================================================
# EXECUTION PREP
# =============================================================================

config_path = RESULTS_ROOT / "experiment_config.json"
save_json(config_path, RUN_CONFIG)
RUN_DEVICE = get_run_device(RESOLVED_GPU_ID)
print(f"Saved config JSON to: {config_path}")
print(f"Sequential execution device: {RUN_DEVICE}")
print(f"Active folds: {ACTIVE_FOLDS}")
print(f"Completed modes currently cached: {list(GLOBAL_MODE_RESULTS.keys())}")


In [ ]:
# =============================================================================
# RUN MODE: MFMC_DTC
# =============================================================================

mode_summary_mfmc_dtc, fold_curves_mfmc_dtc = run_single_mode("mfmc_dtc", gpu_id=RESOLVED_GPU_ID, show_progress=True)
display(pd.DataFrame([mode_summary_mfmc_dtc["summary"]]))


In [ ]:
# =============================================================================
# RUN MODE: PAIRWISE_FMCA
# =============================================================================

mode_summary_pairwise_fmca, fold_curves_pairwise_fmca = run_single_mode("pairwise_fmca", gpu_id=RESOLVED_GPU_ID, show_progress=True)
display(pd.DataFrame([mode_summary_pairwise_fmca["summary"]]))


In [ ]:
# =============================================================================
# RUN MODE: SHUFFLED_DTC
# =============================================================================

mode_summary_shuffled_dtc, fold_curves_shuffled_dtc = run_single_mode("shuffled_dtc", gpu_id=RESOLVED_GPU_ID, show_progress=True)
display(pd.DataFrame([mode_summary_shuffled_dtc["summary"]]))


In [ ]:
# =============================================================================
# FINAL AGGREGATION
# =============================================================================

summary_df = save_aggregate_artifacts()
if summary_df is not None:
    display(summary_df)
